# XGBoost 베이스라인 — Colab 재현
Mac 결과(`results/summary.csv`)와 같은 코드·같은 시드를 Colab에서 돌려 비교합니다. 결과는 `results_colab/`에 따로 저장되어 Mac 결과를 덮어쓰지 않습니다.

**준비:** Elliptic CSV 3개를 구글 드라이브 `내 드라이브/Colab Notebooks/elliptic/` 폴더에 올려 두세요 (`elliptic_txs_features.csv`, `elliptic_txs_edgelist.csv`, `elliptic_txs_classes.csv`).

In [ ]:
# 1) 환경 확인
!python --version
!nvidia-smi -L || echo 'GPU 없음 (XGBoost는 CPU로 돌립니다)'
!nproc

In [ ]:
# 2) 저장소 가져오기 (private이면 토큰 입력)
import getpass, os
if not os.path.exists('2026-2-HAI-ProjectTeam8'):
    tok = getpass.getpass('GitHub 토큰 (public이면 그냥 엔터): ')
    host = f'{tok}@' if tok else ''
    !git clone https://{host}github.com/HanyangTechAI/2026-2-HAI-ProjectTeam8.git
%cd 2026-2-HAI-ProjectTeam8

In [ ]:
# 3) 데이터 복사 (드라이브 -> data/elliptic)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p data/elliptic
!cp "/content/drive/MyDrive/Colab Notebooks/elliptic/"*.csv data/elliptic/
!ls -la data/elliptic

In [ ]:
# 4) 라이브러리 (Colab pip에는 3.4.2가 아직 없어 3.4.x 범위로 설치; 실제 버전을 출력해 기록)
!pip install -q "xgboost>=3.4.1,<3.5" scikit-learn pandas numpy
import xgboost; print('xgboost', xgboost.__version__)

In [ ]:
# 5) 데이터 검증 (모두 OK여야 함)
!python scripts/verify_data.py data/elliptic

In [ ]:
# 6) 베이스라인 실행 (튜닝 30회 + 시드 5개, 결과는 results_colab/ 에 저장)
!python -m baselines.xgb_baseline --data-root data/elliptic --n-jobs -1 --results-dir results_colab

In [ ]:
# 7) Colab 결과 vs Mac 결과 비교
import pandas as pd
pd.set_option('display.width', 200)
cols = ['pr_auc','roc_auc','precision','recall','f1']
m = pd.read_csv('results_colab/metrics/xgb.csv', dtype={'threshold_type': str})
t = m[m.split == 'test']
colab = t.groupby('threshold_type')[cols].agg(['mean','std']).round(4)
print('== Colab =='); print(colab)
print('\n== Colab 시드별 (F1 튀는 시드 확인) =='); print(t[['seed','threshold_type','pr_auc','f1','notes']].round(4).to_string(index=False))
print('\n== Mac (README 기준) ==\nPR-AUC 0.7984 ± 0.0017 / F1@0.5 0.7790 ± 0.0118 / F1@val_opt 0.7836 ± 0.0159')
print('Colab 학습시간 평균(초):', round(t.train_sec.mean(), 3), '| 장치:', t.device.iloc[0])

In [ ]:
# 8) 결과를 드라이브에 백업 (런타임이 끊겨도 보존)
!mkdir -p "/content/drive/MyDrive/Colab Notebooks/elliptic/results_colab"
!cp -r results_colab/* "/content/drive/MyDrive/Colab Notebooks/elliptic/results_colab/"
!ls "/content/drive/MyDrive/Colab Notebooks/elliptic/results_colab"